# Xiangqi AI — Silver → Gold (Policy + Offline Q-Learning)

Reads **only two explicitly selected Silver full runs**: CCPD and Alpha-Beta. Stages copies under /content, verifies SHA-256, runs the backend CLI, validates all output digests, then publishes a new immutable Gold run on Google Drive. This notebook does not implement AI training or rewrite Silver.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
SILVER_ROOT = DRIVE_PROJECT / "data" / "02_silver"
GOLD_ROOT = DRIVE_PROJECT / "data" / "03_gold"
REPO_DIR = Path("/content/ML-Advance-Group")
BACKEND_DIR = REPO_DIR / "project" / "backend"
STAGING_ROOT = Path("/content/xiangqi_gold_stage")

# Chọn chính xác phiên bản full, KHÔNG chọn sample hoặc validated76.
SILVER_RUNS = [
    "ccpd_full_20261107T080132614436Z",
    "alphabeta_full_20261107T074509139310Z",
]
GIT_BRANCH = "feature/silver-to-gold-qlearning-pipeline"
SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
         "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", GIT_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_BRANCH],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)], check=True)
revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Backend revision:", revision)
assert (BACKEND_DIR / "scripts" / "build_gold.py").is_file()

In [ ]:
import hashlib
import shutil
from datetime import datetime, timezone

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_NAME = f"gold_full_{stamp}"
local_inputs = []
for name in SILVER_RUNS:
    source = SILVER_ROOT / name
    target = STAGING_ROOT / name
    if not source.is_dir():
        raise FileNotFoundError(f"Silver run missing: {source}")
    target.mkdir(parents=True, exist_ok=False)
    for filename in ("games.jsonl", "transitions.jsonl"):
        src = source / filename
        dst = target / filename
        if not src.is_file():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        if sha256_file(src) != sha256_file(dst):
            raise IOError(f"SHA-256 mismatch while staging: {src}")
    local_inputs.append(target)
    print("Staged", name)

In [ ]:
import json

LOCAL_OUTPUT = Path("/content/xiangqi_gold_runs") / RUN_NAME
command = [
    sys.executable, "-m", "scripts.build_gold",
    "--silver-dirs", *map(str, local_inputs),
    "--output-dir", str(LOCAL_OUTPUT),
    "--seed", str(SEED),
    "--train-ratio", str(TRAIN_RATIO),
    "--val-ratio", str(VAL_RATIO),
]
subprocess.run(command, cwd=BACKEND_DIR, check=True)
report = json.loads((LOCAL_OUTPUT / "report.json").read_text(encoding="utf-8"))
for filename, expected in report["output_sha256"].items():
    actual = sha256_file(LOCAL_OUTPUT / filename)
    if actual != expected:
        raise IOError(f"Corrupt Gold output {filename}")
assert sum(report["transitions_by_split"].values()) == report["transitions"]
assert sum(report["games_by_split"].values()) == report["accepted_games"]
print("Gold games:", report["accepted_games"], "| transitions:", report["transitions"])
print("By split:", report["transitions_by_split"])

In [ ]:
import os

# Ghi vào thư mục .uploading trước, chỉ đổi tên khi kiểm SHA xong.
GOLD_ROOT.mkdir(parents=True, exist_ok=True)
final_path = GOLD_ROOT / RUN_NAME
temp_path = GOLD_ROOT / f"{RUN_NAME}.uploading"
if final_path.exists() or temp_path.exists():
    raise FileExistsError(f"Gold run already exists: {RUN_NAME}")
temp_path.mkdir(parents=True, exist_ok=False)

for local_file in sorted(LOCAL_OUTPUT.iterdir()):
    target = temp_path / local_file.name
    shutil.copy2(local_file, target)
    if sha256_file(local_file) != sha256_file(target):
        raise IOError(f"Failed Drive upload checksum: {local_file.name}")
if set(p.name for p in temp_path.iterdir()) != set(p.name for p in LOCAL_OUTPUT.iterdir()):
    raise IOError("Gold uploaded files do not match local files")
temp_path.rename(final_path)
print("Published:", final_path)
print("Total transitions:", report["transitions"])
print("Git SHA:", revision)